In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import inspect
import json
from IPython.display import display
from pathlib import Path
import src

## Sequence Info overview
This script is dedicated to giving an overview which sequences were run during the scanning session of a group of scans  <br>
Make sure all participants have a dicominfo tsv file (you can create those by referring to the heudiconv_wrapper.ipynb in this directory). <br>
So it must be run AFTER the heudiconv_wrapper script. <br>

Requirements:
- *dicominfo tsv files* - for each session that should be considered, as generated by heudiconv. See heudiconv_wrapper notebook in this directory. 
- *sorting rules* - to classify scans (modality conditions), this is already specified in the config file in the folder of this script.

Output:
- *sequence_summary.tsv* - a tsv file that indicates which sequences were run for each session

In [ ]:
# Load config
with open("CONFIG_FILE.json", "r") as f:
    cfg = json.load(f)

print("Loaded configuration:")
for k, v in cfg.items():
    print(f"{k}: '{v}'") 

anchor = Path.cwd()   # or notebook dir
root = src.utils.find_root_with_marker(anchor, ".drive_root.txt")
# Load in configs
dicom_root_path_7T = src.utils.combine_paths(root, cfg["RELATIVE_DICOM_ROOT_PATH_7T"])
dicom_root_path_3T = src.utils.combine_paths(root, cfg["RELATIVE_DICOM_ROOT_PATH_3T"])

output_dir_root_7T = src.utils.combine_paths(root, cfg["RELATIVE_OUTPUT_DIR_ROOT_7T"], check_if_exists=False)
output_dir_root_3T = src.utils.combine_paths(root, cfg["RELATIVE_OUTPUT_DIR_ROOT_3T"], check_if_exists=False)

folder_template_7T = cfg["GENERAL_FOLDER_TEMPLATE_7T"].format(dicom_root=dicom_root_path_7T, subject="{subject}") #paste in correct root path, subject stays as placeholder
folder_template_3T = cfg["GENERAL_FOLDER_TEMPLATE_3T"].format(dicom_root=dicom_root_path_3T, subject="{subject}") #paste in correct root path, subject stays as placeholder

dicominfo_tsv_name = cfg["DICOMINFO_TSV_NAME"]

In [ ]:
#group identifer, will be used in filemname for the output files
IDENTIFIER = "MRE_project"

#set up lamd
MODALITY_CONDITIONS = dict()
for modality, keywords in cfg["MODALITY_SEQUENCE_IDENTIFIERS"].items():
    MODALITY_CONDITIONS[modality] = lambda x, kw=keywords: any(k.lower() in x.lower() for k in kw)

MODALITY_CONDITIONS

In [ ]:
res_3T, res_7T = dict(), dict()

for output_dir_root, res in zip([output_dir_root_3T, output_dir_root_7T], [res_3T, res_7T]):
    heudiconv_info_root = output_dir_root

    tsv_file_paths = glob.glob(fr"**{os.sep}{dicominfo_tsv_name.format(subject = "*")}", root_dir= heudiconv_info_root, recursive=True, include_hidden=True)
    tsv_file_paths = [os.path.join(heudiconv_info_root, p) for p in tsv_file_paths]

    print(f"{len(tsv_file_paths)} sessions found in total, with root '{heudiconv_info_root}'.")
    session_IDs = [src.utils.get_ID_from_tsvpath(p) for p in tsv_file_paths]

    #colelct sequence overview (count of sequences of different modalities)
    all_sequence_overviews = list()
    #collect all occuring series descriptions
    all_series_descriptions = list()

    #iterate through found dicom info tsvs
    for tsv_file_path in tsv_file_paths:
        cur_id = src.utils.get_ID_from_tsvpath(tsv_file_path)
        subject_res = {"Session_ID": cur_id}
        
        tsv_file = pd.read_csv(tsv_file_path, sep='\t', encoding='latin-1', on_bad_lines='skip')

        #apply all modality checks and sum over how many matching sequences were found for the current subject
        for modality, condition in MODALITY_CONDITIONS.items():
            modality_count = sum(tsv_file["series_description"].apply(condition))
            subject_res[modality] = modality_count
        all_sequence_overviews.append(subject_res)

        series_description = tsv_file["series_description"].values
        all_series_descriptions.extend(series_description)

    description_counts = pd.Series(all_series_descriptions).value_counts()
    description_counts = description_counts.reset_index()
    description_counts.columns = ["series_description", "count"]

    res["sequence_overview"] = pd.DataFrame(all_sequence_overviews)
    res["description_counts"] = description_counts


In [ ]:
display(res_3T["sequence_overview"])
display(res_3T["description_counts"])

In [ ]:
display(res_7T["sequence_overview"])
display(res_7T["description_counts"])

## Write to system

summary file

In [ ]:
overwrite = True
seq_overview_7T_out_path = os.path.join(output_dir_root_7T, "sequence_summary_7T.tsv")
src.utils.write_df_as_tsv(res_7T["sequence_overview"], seq_overview_7T_out_path, overwrite=overwrite)

seq_overview_3T_out_path = os.path.join(output_dir_root_3T, "sequence_summary_3T.tsv")
src.utils.write_df_as_tsv(res_3T["sequence_overview"], seq_overview_3T_out_path, overwrite=overwrite)